# Declarative Data Visualisation
This strategy is "declarative" in nature. Instead of writing a "program" in the traditional sense, we write instead a JSON "spec" which is fed to an interpreter. The interpreter deals with the tedious mathematics, getting lines on screens etc. Libraries which follow this approach include, but are not limited to 

- [Vega](https://vega.github.io)
- [Vega lite](https://vega.github.io/vega-lite)
- [Apache E-charts](https://echarts.apache.org/en/index.html)
- [Plotly](https://plotly.com/graphing-libraries/)

Co-incidentally, all of these libraries, use JSON as the "spec format". There's actually nothing really special about the choice of vega here... other than it's convieniently built right into Jupyter and I had a good experience with it :-).

In [124]:
import $ivy.`com.lihaoyi::upickle:4.4.2`
import $ivy.`sh.almond:json-api-jackson_3:0.14.2`
import java.util.Base64
import com.fasterxml.jackson.databind.ObjectMapper
import almond.api.AlmondJackson.Extensions.{*, given}

lazy val specStr = """{
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "description": "A simple bar chart with embedded data.",
  "width" : 300,
  "height": 300,
  "data": {
    "values": [
      {"a": "A", "b": 28}, {"a": "B", "b": 55}, {"a": "C", "b": 43}
    ]
  },
  "mark": "bar",
  "encoding": {
    "x": {"field": "a", "type": "nominal", "axis": {"labelAngle": 0}},
    "y": {"field": "b", "type": "quantitative"}
  }
}"""

lazy val spec = ujson.read(specStr)

spec("data")("values") = ujson.Arr(
  ujson.Obj("a" -> "Fabulous", "b" -> 20),
  ujson.Obj("a" -> "Charting", "b" -> 10),
  ujson.Obj("a" -> "Potential", "b" -> 100)
)

kernel.publish.displayDataObject(
  Map(("application/vnd.vegalite.v4+json", ujson.write(spec)))
)

import $ivy.$                           

import $ivy.$                                    

import java.util.Base64

import com.fasterxml.jackson.databind.ObjectMapper

import almond.api.AlmondJackson.Extensions.{*, given}


specStr: String = <lazy>
spec: Value = <lazy>

In [125]:
kernel.publish.html("<div id=\"viz\"> a Div</div>")
kernel.publish.html("""            <style>
                #viz {
                    width: 100%;
                    height: 1000px;
                    display: flex;
                }
            </style>""")
kernel.publish.js(raw"""
let outputDiv = document.getElementById("viz");
outputDiv.innerHTML = "gotcha";

function showError(err) {
    outputDiv.innerHTML = outputDiv.innerHTML + "<b style='color:red'>Error: " + err + "</b>";
}


function maybeloadLibrary(url) {
  return new Promise((resolve, reject) => {
    outputDiv.innerHTML = outputDiv.innerHTML + "<p>loading " + "https://cdn.jsdelivr.net/npm/" + url + "</p>";
    const url_enriched = "https://cdn.jsdelivr.net/npm/" + url;
    let script = document.createElement('script');
    script.src = url_enriched;
    script.async = false;
    // script.onload = () => {return resolve(url_enriched);}
    // script.onerror = () => reject("Failed to load library: " + url_enriched);
    document.head.appendChild(script);
  });
}

outputDiv.innerHTML = "got here";

maybeloadLibrary("vega@5.25.0/build/vega.js")
maybeloadLibrary("vega-lite@5.20.1/build/vega-lite.js")
maybeloadLibrary("vega-embed@6.25.0/build/vega-embed.js")
outputDiv.innerHTML = outputDiv.innerHTML + "<p>all libraries loaded</p>"
const spec = ${ujson.write(spec)};

vegaEmbed('#viz', spec, {
                renderer: "svg", // renderer (canvas or svg)
                container: "#vis", // parent DOM container
                hover: true, // enable hover processing
                actions: {
                  editor : true
                }
            }).then(function(result) {
              outputDiv.innerHTML = outputDiv.innerHTML + "<p>Visualization rendered</p>";
            }).catch(function(err) {
              showError(err);
            });

""")

a Div

In [ ]:
// Clean version - temporarily disables AMD so libs register as globals
val vizId2 = s"viz-${java.util.UUID.randomUUID().toString.replace("-", "")}"

kernel.publish.html(s"""<div id="$vizId2"></div>""")
kernel.publish.html(s"""<style> #$vizId2.vega-embed {width:100%; display:flex;} #$vizId2.vega-embed details, #$vizId2.vega-embed summary {position: relative;} </style>""")

kernel.publish.js(s"""
(function() {
  var outputDiv = document.getElementById("$vizId2");

  // Temporarily hide AMD so libraries register as globals
  var oldDefine = window.define;
  window.define = undefined;

  function loadScript(url) {
    return new Promise(function(resolve, reject) {
      var s = document.createElement('script');
      s.src = url;
      s.async = true;
      s.onload = resolve;
      s.onerror = function() { reject(new Error("Failed to load " + url)); };
      document.head.appendChild(s);
    });
  }

  loadScript("https://cdn.jsdelivr.net/npm/vega@5")
    .then(function() { return loadScript("https://cdn.jsdelivr.net/npm/vega-lite@5"); })
    .then(function() { return loadScript("https://cdn.jsdelivr.net/npm/vega-embed@6"); })
    .then(function() {
      window.define = oldDefine;  // Restore AMD
      vegaEmbed(outputDiv, ${ujson.write(spec)}, {mode: "vega-lite"});
    })
    .catch(function(err) {
      window.define = oldDefine;  // Restore AMD even on error
      outputDiv.innerHTML = "<b style='color:red'>Error: " + err.message + "</b>";
    });
})();
""")

In [143]:
val vizId = s"viz-${java.util.UUID.randomUUID().toString.replace("-", "")}"

kernel.publish.html(s"""<div id="${vizId}-debug" style="font-family: monospace; font-size: 12px;"></div>
<div id="$vizId"></div>""")

kernel.publish.js(s"""
var debug = document.getElementById("${vizId}-debug");
var outputDiv = document.getElementById("$vizId");

function log(msg) {
  debug.innerHTML += "<p>" + msg + "</p>";
}

function loadScriptInline(url) {
  return fetch(url)
    .then(function(response) {
      if (!response.ok) throw new Error("HTTP " + response.status);
      return response.text();
    })
    .then(function(code) {
      log("Fetched: " + url);
      // Execute with AMD disabled so libs register as globals
      var fn = new Function('define', code);
      fn(undefined);  // Pass undefined as 'define' to disable AMD detection
      log("Executed: " + url);
    });
}

log("Script started - fetching libraries (AMD disabled)");

loadScriptInline("https://cdn.jsdelivr.net/npm/vega@5")
  .then(function() {
    log("vega type: " + typeof vega);
    return loadScriptInline("https://cdn.jsdelivr.net/npm/vega-lite@5");
  })
  .then(function() {
    log("vegaLite type: " + typeof vegaLite);
    return loadScriptInline("https://cdn.jsdelivr.net/npm/vega-embed@6");
  })
  .then(function() {
    log("All libraries loaded!");
    log("vegaEmbed type: " + typeof vegaEmbed);
    log("window.vegaEmbed type: " + typeof window.vegaEmbed);

    var embed = vegaEmbed || window.vegaEmbed;
    var spec = ${ujson.write(spec)};
    embed(outputDiv, spec, {mode: "vega-lite"})
      .then(function() { log("Chart rendered!"); })
      .catch(function(err) { log("vegaEmbed ERROR: " + err.message); });
  })
  .catch(function(err) {
    log("Error: " + err.message);
  });
""")

vizId: String = "viz-7429fbd163bb407ab8f42da3f5969673"

In [142]:
// Clean version - temporarily disables AMD so libs register as globals
val vizId2 = s"viz-${java.util.UUID.randomUUID().toString.replace("-", "")}"

kernel.publish.html(s"""<div id="$vizId2"></div>""")
kernel.publish.html(s"""<style> #$vizId2.vega-embed {width:100%; display:flex;} #$vizId2.vega-embed details, #$vizId2.vega-embed summary {position: relative;} </style>""")

kernel.publish.js(s"""
(function() {
  var outputDiv = document.getElementById("$vizId2");

  // Temporarily hide AMD so libraries register as globals
  var oldDefine = window.define;
  window.define = undefined;

  function loadScript(url) {
    return new Promise(function(resolve, reject) {
      var s = document.createElement('script');
      s.src = url;
      s.async = true;
      s.onload = resolve;
      s.onerror = function() { reject(new Error("Failed to load " + url)); };
      document.head.appendChild(s);
    });
  }

  loadScript("https://cdn.jsdelivr.net/npm/vega@5")
    .then(function() { return loadScript("https://cdn.jsdelivr.net/npm/vega-lite@5"); })
    .then(function() { return loadScript("https://cdn.jsdelivr.net/npm/vega-embed@6"); })
    .then(function() {
      window.define = oldDefine;  // Restore AMD
      vegaEmbed(outputDiv, ${ujson.write(spec)}, {mode: "vega-lite"});
    })
    .catch(function(err) {
      window.define = oldDefine;  // Restore AMD even on error
      outputDiv.innerHTML = "<b style='color:red'>Error: " + err.message + "</b>";
    });
})();
""")

vizId2: String = "viz-988e5d9e213148049942965c24d4cdf8"

In [146]:
// Version using dynamic imports (await import)
val vizId3 = s"viz-${java.util.UUID.randomUUID().toString.replace("-", "")}"

kernel.publish.html(s"""<div id="$vizId3"></div>""")
kernel.publish.html(s"""<style> #$vizId3.vega-embed {width:100%; display:flex;} #$vizId3.vega-embed details, #$vizId3.vega-embed summary {position: relative;} </style>""")

kernel.publish.js(s"""
(async function() {
  var outputDiv = document.getElementById("$vizId3");

  try {
    const vega = await import("https://cdn.jsdelivr.net/npm/vega@5/+esm");
    const vegaLite = await import("https://cdn.jsdelivr.net/npm/vega-lite@5/+esm");
    const vegaEmbed = await import("https://cdn.jsdelivr.net/npm/vega-embed@6/+esm");

    await vegaEmbed.default(outputDiv, ${ujson.write(spec)}, {mode: "vega-lite"});
  } catch (err) {
    outputDiv.innerHTML = "<b style='color:red'>Error: " + err.message + "</b>";
  }
})();
""")

vizId3: String = "viz-8a56423cf00641239ac6eb31ee938eb2"

Now we changed the data! By mutating a JSON object... if you like immutable scala, maybe that part was painful on the eyeballs :-). 

However, I would not rush to discard this as a plotting solution - looks nasty, but surprisingly, solves the general problem. By reading the vega docs, you can plot _anything_ (that vega can plot) like this. I hesitate to point out that "making the correct JSON" is a testable problem... 

Also, this strategy trivially cross compiles to scala JS. It's easy to setup if you are willing to totally plagarise [SJRDs example](https://github.com/sjrd/scalajs-sbt-vite-laminar-chartjs-example), and change it slightly to [bundle vega embed](https://github.com/Quafadas/scalajs-sbt-vite-laminar-chartjs-example) and write a [simple facade](https://github.com/Quafadas/dedav4s/blob/861c3fa38f41084f9d2e1ea168da40aab22eccf5/core/js/src/main/scala/viz/vega/facades/VegaEmbed.scala#L83). It's also pretty easy to embed in something like play! if you have static routes. Pump out the JSON into your template as a variable, include the vega libraries in the header of the page. Simples.

So you already have "full stack" embedabble plots, and the only library you need, is ~~ujson~~ any Json library. 

With ujson, it's easy to mess around in Jupyter / ammonite JVM land then use the _exact same code_ to publish in your full stack project. 

I claim that, if the use case is a single, complex chart, and you're willing to
- solve the meta-problems of getting vega libs in the right place
- write the facade
- use mutable stuff

Then we're done. Surpisingly, I claim this solution is significantly less sh*t than it would appear at first blush - I've used it to visualise (on the fly) the [high dimensional characsterics](https://vega.github.io/vega/examples/brushing-scatter-plots/) of a portfolio of reinsurance contracts...